# Tensile Test Analysis: Engineering Stress-Strain

Processes a raw tensile-test CSV (`Load (N)`, `Displacement (mm)`) and produces:

1. Cleaned data (NaN removal, initial load/displacement offset zeroed)
2. Engineering stress (MPa) and strain (mm/mm)
3. Young's modulus (automatic linear-region detection)
4. 0.2 % offset yield strength
5. Ultimate tensile strength (UTS) and elongation at fracture
6. A publication-quality stress-strain plot

**How to use:** run the cells from top to bottom (*Runtime > Run all*). In the *Inputs* cell, set the gauge length and cross-sectional area, then upload your CSV when prompted. To try the notebook first, tick `USE_SAMPLE_DATA`.

In [ ]:
import logging
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## 1. Loading and cleaning

In [ ]:
LOAD_COL = "Load (N)"
DISP_COL = "Displacement (mm)"
STRESS_COL = "Engineering Stress (MPa)"
STRAIN_COL = "Engineering Strain (mm/mm)"

log = logging.getLogger("tensile")


def load_raw_data(path, sep=",") -> pd.DataFrame:
    """Read the CSV and return only the two required columns."""
    if not hasattr(path, "read"):  # a path, not an uploaded/open file object
        path = Path(path)
        if not path.is_file():
            raise FileNotFoundError(f"File not found: {path}")

    # utf-8-sig strips the BOM that many test-machine exports include
    df = pd.read_csv(path, sep=sep, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()  # tolerate stray whitespace in headers

    missing = {LOAD_COL, DISP_COL} - set(df.columns)
    if missing:
        raise KeyError(
            f"Missing required column(s): {sorted(missing)}. "
            f"Found: {list(df.columns)}"
        )
    return df[[LOAD_COL, DISP_COL]].copy()


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce to numeric and drop NaN / inf / non-numeric rows."""
    n_start = len(df)

    # Non-numeric entries (e.g. 'N/A', units text, '---') become NaN
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna().reset_index(drop=True)

    n_dropped = n_start - len(df)
    if n_dropped:
        log.warning("Dropped %d invalid row(s) out of %d.", n_dropped, n_start)
    if df.empty:
        raise ValueError("No valid data remains after cleaning.")
    return df

## 2. Offset zeroing and engineering stress / strain

- Engineering stress: $\sigma = F / A_0$ (N/mm² = MPa)
- Engineering strain: $\varepsilon = \Delta L / L_0$ (mm/mm)

In [ ]:
def zero_offset(df: pd.DataFrame, n_baseline: int = 5):
    """
    Shift both columns so the test starts at (0, 0).

    The offset is the median of the first `n_baseline` points rather than
    just the first row, so a single noisy reading doesn't skew the baseline.
    Returns the zeroed DataFrame and the offsets that were removed.
    """
    n = max(1, min(n_baseline, len(df)))
    offsets = df.iloc[:n].median()

    zeroed = df - offsets

    # Sanity check: warn if the load offset is a notable fraction of peak load
    peak = zeroed[LOAD_COL].abs().max()
    if peak > 0 and abs(offsets[LOAD_COL]) > 0.05 * peak:
        log.warning(
            "Initial load offset (%.3f N) is >5%% of peak load. This may be "
            "preload or slack rather than a simple machine offset - verify.",
            offsets[LOAD_COL],
        )

    log.info(
        "Zeroed offsets removed: load = %.4f N, displacement = %.4f mm",
        offsets[LOAD_COL],
        offsets[DISP_COL],
    )
    return zeroed, offsets


def to_engineering(df: pd.DataFrame, l0_mm: float, a0_mm2: float) -> pd.DataFrame:
    """
    Convert load-displacement data to engineering stress-strain.

    Parameters
    ----------
    l0_mm  : initial gauge length L0 [mm]
    a0_mm2 : initial (original) cross-sectional area A0 [mm^2]
    """
    if l0_mm <= 0 or a0_mm2 <= 0:
        raise ValueError("L0 and A0 must both be greater than zero.")

    out = pd.DataFrame(index=df.index)

    # Engineering stress:  sigma = F / A0
    #   F  = instantaneous applied load [N]
    #   A0 = ORIGINAL cross-sectional area [mm^2] (not the instantaneous area,
    #        which is what distinguishes engineering from true stress)
    #   Unit check: 1 N / 1 mm^2 = 1 MPa, so no unit conversion is needed.
    out[STRESS_COL] = df[LOAD_COL] / a0_mm2

    # Engineering strain:  epsilon = (L - L0) / L0 = delta_L / L0
    #   delta_L = change in length = measured displacement [mm]
    #   L0      = ORIGINAL gauge length [mm]
    #   Strain is dimensionless (mm/mm); multiply by 100 for % elongation.
    out[STRAIN_COL] = df[DISP_COL] / l0_mm

    return out

## 3. Young's modulus and 0.2 % offset yield strength

The linear region is found by sliding a window (15 % of UTS in stress) over the 10-80 % of UTS band, keeping windows with R² ≥ 0.995, choosing the steepest one, then growing it outward while points stay within 1 % of UTS of the fitted line.

In [ ]:
@dataclass
class ElasticFit:
    """Result of the linear-elastic-region fit."""
    modulus_mpa: float      # slope E of stress vs strain [MPa]
    intercept_mpa: float    # stress at zero strain of the fitted line [MPa]
    r_squared: float        # goodness of fit over the final region
    start_idx: int          # first row of the linear region
    end_idx: int            # last row of the linear region

    @property
    def modulus_gpa(self) -> float:
        return self.modulus_mpa / 1000.0


def _linear_fit(x: np.ndarray, y: np.ndarray):
    """Least-squares line y = m*x + b. Returns (m, b, R^2)."""
    m, b = np.polyfit(x, y, 1)
    ss_res = np.sum((y - (m * x + b)) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return m, b, r2


def find_youngs_modulus(
    df: pd.DataFrame,
    min_points: int = 10,
    window_stress_frac: float = 0.15,
    r2_min: float = 0.995,
    stress_bounds: tuple = (0.10, 0.80),
    deviation_tol: float = 0.01,
) -> ElasticFit:
    """
    Automatically locate the linear elastic region and fit Young's modulus.

    Plain-English summary of the algorithm
    --------------------------------------
    1. Keep only data up to the peak stress (UTS). Everything after it is
       necking / fracture and can never be part of the elastic region.
    2. Only look between 10 % and 80 % of UTS (stress_bounds). Below 10 % the
       curve is dominated by toe-region slack and noise; the elastic region
       is not expected to extend past ~80 % of UTS on any usual metal.
    3. Slide a window along that range and fit a straight line in each
       window. Each window spans a fixed stress range (15 % of UTS) rather
       than a fixed number of points, so the method works the same whether
       the data was logged at 5 Hz or 500 Hz. Keep windows that are very
       straight (R^2 >= r2_min).
    4. Of those straight windows, pick the STEEPEST. Elastic deformation is
       the stiffest part of the curve: the toe region (slack take-up) is
       flatter, and plastic yielding is flatter too.
    5. Grow that window outward in both directions, point by point, as long
       as the data stays within `deviation_tol` (1 % of UTS) of the fitted
       line. Growth backwards stops where the toe region begins to curve
       away; growth forwards stops where the curve departs from linearity
       (the proportional limit). Those stopping points are the start and end
       of the linear region.
    6. Refit one final line over the full grown region. Its slope is E.
    """
    stress = df[STRESS_COL].to_numpy(dtype=float)
    strain = df[STRAIN_COL].to_numpy(dtype=float)

    # Step 1: truncate at peak stress
    peak = int(np.argmax(stress))
    uts = stress[peak]
    if uts <= 0:
        raise ValueError("Peak stress is not positive; cannot find elastic region.")

    # Step 2: restrict the search to the allowed stress band (pre-peak only)
    lo_frac, hi_frac = stress_bounds
    lo_hits = np.where(stress[: peak + 1] >= lo_frac * uts)[0]
    hi_hits = np.where(stress[: peak + 1] >= hi_frac * uts)[0]
    lo_idx, hi_idx = int(lo_hits[0]), int(hi_hits[0])
    span = hi_idx - lo_idx + 1
    if span < min_points:
        raise ValueError(
            f"Only {span} points lie between {lo_frac:.0%} and {hi_frac:.0%} of UTS "
            f"(need {min_points}). Data may be too sparse in the elastic region."
        )

    # Step 3 & 4: slide the window, keep straight ones, choose the steepest
    win_span = window_stress_frac * uts
    best = None  # (slope, intercept, window_start, window_end)
    for s in range(lo_idx, hi_idx):
        reach = np.where(stress[s : hi_idx + 1] >= stress[s] + win_span)[0]
        if reach.size == 0:
            break  # window would run past the allowed stress band
        e = s + int(reach[0])
        if e - s + 1 < min_points:
            continue
        m, b, r2 = _linear_fit(strain[s : e + 1], stress[s : e + 1])
        if r2 >= r2_min and m > 0 and (best is None or m > best[0]):
            best = (m, b, s, e)
    if best is None:
        raise ValueError(
            f"No window reached R^2 >= {r2_min}. Data may be too noisy or too "
            "sparse; try lowering r2_min or min_points."
        )
    m, b, w_start, w_end = best

    # Step 5: grow the region outward while points stay near the fitted line
    tol = deviation_tol * uts
    resid = stress - (m * strain + b)
    start, end = w_start, w_end
    while start > 0 and abs(resid[start - 1]) <= tol:
        start -= 1
    while end < peak and abs(resid[end + 1]) <= tol:
        end += 1

    # Step 6: final refit over the whole region
    m, b, r2 = _linear_fit(strain[start : end + 1], stress[start : end + 1])
    log.info(
        "Linear region: rows %d-%d (%.1f-%.1f MPa), E = %.1f GPa, R^2 = %.5f",
        start, end, stress[start], stress[end], m / 1000.0, r2,
    )
    return ElasticFit(m, b, r2, start, end)


def calculate_yield_strength(
    df: pd.DataFrame, fit: ElasticFit, offset: float = 0.002
):
    """
    0.2 % offset yield strength.

    Method
    ------
    The fitted elastic line is  sigma = E*eps + b.  Its strain-axis intercept
    is eps0 = -b/E (this automatically absorbs any residual toe offset).
    The offset line is the elastic line shifted right by `offset` strain
    (0.002 = 0.2 %):

        sigma_offset(eps) = E * (eps - eps0 - offset) = E*eps + b - E*offset

    Yield is where the measured curve first falls onto this line. We track
    diff = sigma_measured - sigma_offset. It starts positive (the curve sits
    above the shifted line) and the first sign change to <= 0 after the
    linear region is the crossing, which is located by linear interpolation
    between the two bracketing points.

    Only data up to peak stress is searched, so the post-fracture load drop
    cannot be mistaken for yielding. If no crossing occurs by peak stress
    (e.g. a brittle specimen), (nan, nan) is returned.

    Returns
    -------
    (yield_strength_mpa, yield_strain)
    """
    stress = df[STRESS_COL].to_numpy(dtype=float)
    strain = df[STRAIN_COL].to_numpy(dtype=float)
    peak = int(np.argmax(stress))

    E, b = fit.modulus_mpa, fit.intercept_mpa
    diff = stress - (E * strain + b - E * offset)

    # First point after the linear region where the curve is on/below the offset line
    start = fit.end_idx
    below = np.where(diff[start : peak + 1] <= 0)[0]
    if below.size == 0:
        log.warning("Curve never crosses the %.1f%% offset line before peak "
                    "stress; yield strength not defined.", offset * 100)
        return float("nan"), float("nan")

    i = start + int(below[0])
    if i == 0 or diff[i - 1] <= 0:
        return float(stress[i]), float(strain[i])

    # Linear interpolation between points i-1 (above line) and i (on/below)
    t = diff[i - 1] / (diff[i - 1] - diff[i])
    ys = stress[i - 1] + t * (stress[i] - stress[i - 1])
    ye = strain[i - 1] + t * (strain[i] - strain[i - 1])
    return float(ys), float(ye)

## 4. UTS, fracture point and elongation

In [ ]:
def calculate_uts(df: pd.DataFrame):
    """
    Ultimate tensile strength.

    UTS = F_max / A0 = the maximum engineering stress on the curve.
    Also returns the strain at that point (the uniform elongation, i.e. the
    strain at which necking begins in a ductile metal).

    Returns
    -------
    (uts_mpa, strain_at_uts)
    """
    stress = df[STRESS_COL].to_numpy(dtype=float)
    strain = df[STRAIN_COL].to_numpy(dtype=float)
    idx = int(np.argmax(stress))
    return float(stress[idx]), float(strain[idx])


def find_fracture_index(
    df: pd.DataFrame, drop_frac: float = 0.5, step_frac: float = 0.02
) -> int:
    """
    Locate the fracture point: the last data point before the sudden load drop.

    Plain-English summary
    ---------------------
    1. After the peak stress, find the first point where stress has collapsed
       below `drop_frac` (50 %) of UTS. That only happens when the specimen
       breaks; ordinary necking never takes stress down that far.
    2. The fall from full load to ~zero can span several data points, so walk
       backwards from there while each step is a steep drop (more than
       `step_frac` = 2 % of UTS per point). Stop where the curve is
       back to its normal gentle necking slope. That point is fracture.
    3. If the stress never collapses (the test was stopped before failure),
       the last data point is returned and a warning is logged.
    """
    stress = df[STRESS_COL].to_numpy(dtype=float)
    peak = int(np.argmax(stress))
    uts = stress[peak]

    collapsed = np.where(stress[peak:] < drop_frac * uts)[0]
    if collapsed.size == 0:
        log.warning(
            "No sudden load drop found after peak stress; using the last data "
            "point as 'fracture'. The test may have been stopped before failure."
        )
        return len(stress) - 1

    idx = peak + int(collapsed[0]) - 1  # last point still above the threshold
    while idx > peak and (stress[idx - 1] - stress[idx]) > step_frac * uts:
        idx -= 1
    return idx


def calculate_elongation_at_fracture(
    df: pd.DataFrame, fit: ElasticFit, fracture_idx: int
):
    """
    Percentage elongation at fracture.

    Total elongation   = eps_f * 100
        eps_f = (L_f - L0) / L0 at the fracture point; includes the elastic
        part, which springs back when the specimen breaks.

    Plastic elongation = (eps_f - sigma_f / E) * 100
        This is the 'elongation after fracture' that you would measure by
        fitting the broken halves together (ASTM E8 / ISO 6892-1). It removes
        the elastic recovery, which unloads along a line of slope E from the
        fracture point (sigma_f = stress at fracture).

    Note: both are measured from the zeroed data, so any toe-region slack
    that was not removed by the baseline zeroing is included.

    Returns
    -------
    (total_elongation_pct, plastic_elongation_pct)
    """
    eps_f = float(df[STRAIN_COL].iloc[fracture_idx])
    sig_f = float(df[STRESS_COL].iloc[fracture_idx])
    total = eps_f * 100.0
    plastic = (eps_f - sig_f / fit.modulus_mpa) * 100.0
    return total, plastic


@dataclass
class TensileProperties:
    """All extracted properties for one test."""
    fit: ElasticFit
    yield_strength_mpa: float
    yield_strain: float
    uts_mpa: float
    uts_strain: float            # uniform elongation (strain at UTS)
    fracture_idx: int
    fracture_strain: float
    fracture_stress_mpa: float
    elongation_total_pct: float
    elongation_plastic_pct: float


def analyze_tensile_properties(df: pd.DataFrame) -> TensileProperties:
    """Run the full property extraction on an engineering stress-strain DataFrame."""
    fit = find_youngs_modulus(df)
    ys, ye = calculate_yield_strength(df, fit)
    uts, eps_uts = calculate_uts(df)
    fi = find_fracture_index(df)
    total, plastic = calculate_elongation_at_fracture(df, fit, fi)
    return TensileProperties(
        fit=fit,
        yield_strength_mpa=ys,
        yield_strain=ye,
        uts_mpa=uts,
        uts_strain=eps_uts,
        fracture_idx=fi,
        fracture_strain=float(df[STRAIN_COL].iloc[fi]),
        fracture_stress_mpa=float(df[STRESS_COL].iloc[fi]),
        elongation_total_pct=total,
        elongation_plastic_pct=plastic,
    )

## 5. Plot

In [ ]:
def plot_stress_strain(
    df: pd.DataFrame,
    props: TensileProperties,
    save_path=None,
    show: bool = False,
    offset: float = 0.002,
    title: str = None,
    inset: bool = True,
    inset_rect=(0.57, 0.09, 0.39, 0.28),
):
    """
    Publication-style engineering stress-strain curve.

    Shows: the experimental curve (up to fracture), the fitted elastic line,
    the 0.2 % offset line, and annotated yield, UTS and fracture points.
    With inset=True, a zoomed view of the yield region is added, because at
    full-strain scale the elastic region is squeezed into a near-vertical
    sliver and the offset line is hard to tell apart from the elastic line.
    `inset_rect` = [x0, y0, width, height] in axes fractions; move it if it
    collides with your curve.
    """
    import matplotlib

    import matplotlib.pyplot as plt
    from matplotlib.ticker import AutoMinorLocator

    style = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 10,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "axes.linewidth": 0.8,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
    }

    # Colour-blind-safe (Okabe-Ito) palette
    c_curve, c_elastic, c_offset = "#0072B2", "#009E73", "#D55E00"
    c_yield, c_uts, c_frac = "#D55E00", "#CC79A7", "#000000"

    fit = props.fit
    E, b = fit.modulus_mpa, fit.intercept_mpa
    eps0 = -b / E  # strain where the elastic line crosses zero stress

    # Plot the curve only up to fracture (the post-fracture drop is not material behaviour)
    strain = df[STRAIN_COL].to_numpy()[: props.fracture_idx + 1]
    stress = df[STRESS_COL].to_numpy()[: props.fracture_idx + 1]

    has_yield = np.isfinite(props.yield_strength_mpa)
    sigma_top = 1.15 * props.yield_strength_mpa if has_yield else 0.9 * props.uts_mpa
    eps_top = (sigma_top - b) / E
    el_x, el_y = np.array([eps0, eps_top]), np.array([0.0, sigma_top])
    off_x, off_y = el_x + offset, el_y

    def draw_lines(ax, lw_curve=1.6, lw_lines=1.1):
        ax.plot(strain, stress, color=c_curve, lw=lw_curve, zorder=3)
        ax.plot(el_x, el_y, color=c_elastic, lw=lw_lines, ls="-.", zorder=2)
        ax.plot(off_x, off_y, color=c_offset, lw=lw_lines, ls="--", zorder=2)

    with plt.rc_context(style):
        fig, ax = plt.subplots(figsize=(6.5, 4.5))

        # Main curve and construction lines (legend handles come from these)
        ax.plot(strain, stress, color=c_curve, lw=1.6, zorder=3,
                label="Experimental data")
        ax.plot(el_x, el_y, color=c_elastic, lw=1.1, ls="-.", zorder=2,
                label=f"Linear elastic fit ($E$ = {fit.modulus_gpa:.1f} GPa)")
        ax.plot(off_x, off_y, color=c_offset, lw=1.1, ls="--", zorder=2,
                label=f"{offset * 100:.1f} % offset line")

        box = dict(boxstyle="round,pad=0.3", fc="white", ec="0.6", lw=0.6, alpha=0.95)
        arrow = dict(arrowstyle="-|>", color="0.25", lw=0.8, shrinkA=0, shrinkB=5)

        # Yield point
        if has_yield:
            ax.plot(props.yield_strain, props.yield_strength_mpa, "o", ms=7,
                    mfc=c_yield, mec="black", mew=0.8, zorder=5)
            ax.annotate(
                f"Yield point\n$\\sigma_y$ = {props.yield_strength_mpa:.0f} MPa",
                xy=(props.yield_strain, props.yield_strength_mpa),
                xytext=(45, -50), textcoords="offset points",
                ha="left", va="top", bbox=box, arrowprops=arrow, zorder=6)

        # Ultimate tensile strength
        ax.plot(props.uts_strain, props.uts_mpa, "s", ms=7, mfc=c_uts,
                mec="black", mew=0.8, zorder=5)
        ax.annotate(
            f"UTS\n$\\sigma_{{UTS}}$ = {props.uts_mpa:.0f} MPa",
            xy=(props.uts_strain, props.uts_mpa),
            xytext=(0, 38), textcoords="offset points",
            ha="center", va="bottom", bbox=box, arrowprops=arrow, zorder=6)

        # Fracture point
        ax.plot(props.fracture_strain, props.fracture_stress_mpa, "X", ms=9,
                mfc=c_frac, mec="white", mew=0.8, zorder=5)
        ax.annotate(
            f"Fracture\n$\\varepsilon_f$ = {props.elongation_total_pct:.1f} %",
            xy=(props.fracture_strain, props.fracture_stress_mpa),
            xytext=(-40, -32), textcoords="offset points",
            ha="right", va="top", bbox=box, arrowprops=arrow, zorder=6)

        # Axes, grid, ticks
        ax.set_xlabel("Engineering Strain (mm/mm)")
        ax.set_ylabel("Engineering Stress (MPa)")
        ax.set_xlim(left=min(0.0, eps0), right=props.fracture_strain * 1.05)
        ax.set_ylim(0, props.uts_mpa * 1.22)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.set_axisbelow(True)
        ax.grid(True, which="major", color="0.85", lw=0.6)
        ax.grid(True, which="minor", color="0.93", lw=0.4)
        ax.tick_params(which="both", direction="in", top=True, right=True)
        if title:
            ax.set_title(title)

        ax.legend(loc="upper left", frameon=True, framealpha=0.95,
                  edgecolor="0.7", fancybox=False, borderpad=0.6)

        # Zoomed inset on the elastic / yield region
        if inset and has_yield:
            axins = ax.inset_axes(list(inset_rect))
            draw_lines(axins, lw_curve=1.3, lw_lines=0.9)
            axins.plot(props.yield_strain, props.yield_strength_mpa, "o", ms=5,
                       mfc=c_yield, mec="black", mew=0.6, zorder=5)
            x_hi = props.yield_strain + 1.5 * offset
            mask = strain <= x_hi
            y_hi = max(sigma_top, stress[mask].max() if mask.any() else sigma_top)
            axins.set_xlim(min(0.0, eps0), x_hi)
            axins.set_ylim(0, y_hi * 1.05)
            axins.set_title("Yield region (zoom)", fontsize=8, pad=3)
            axins.tick_params(labelsize=7, direction="in")
            axins.grid(True, color="0.88", lw=0.4)
            axins.set_axisbelow(True)
            axins.set_facecolor("white")
            ax.indicate_inset_zoom(axins, edgecolor="0.4", lw=0.7)

        fig.tight_layout()
        if save_path:
            fig.savefig(save_path, dpi=300, bbox_inches="tight")
            log.info("Saved plot to %s", save_path)
        if show:
            plt.show()
    return fig, ax

## 6. Pipeline

In [ ]:
def process_tensile_file(
    path, l0_mm: float, a0_mm2: float, n_baseline: int = 5, sep=","
) -> pd.DataFrame:
    """Full pipeline: load -> clean -> zero -> engineering stress/strain."""
    df = load_raw_data(path, sep=sep)
    df = clean_data(df)
    df, _ = zero_offset(df, n_baseline=n_baseline)
    return to_engineering(df, l0_mm, a0_mm2)

## Inputs

Set the specimen geometry, then run the cell. Tick `USE_SAMPLE_DATA` to test with a synthetic dataset instead of uploading a file.

In [ ]:
#@title Specimen geometry and options
GAUGE_LENGTH_MM = 50.0       #@param {type:"number"}
AREA_MM2 = 78.5              #@param {type:"number"}
BASELINE_POINTS = 5          #@param {type:"integer"}
USE_SAMPLE_DATA = False      #@param {type:"boolean"}

if GAUGE_LENGTH_MM <= 0 or AREA_MM2 <= 0:
    raise ValueError("Gauge length and area must both be greater than zero.")

if USE_SAMPLE_DATA:
    # Synthetic mild-steel-like test with offsets, a NaN, toe region and necking
    rng = np.random.default_rng(2)
    E, ey, toe = 200000.0, 0.0015, 0.0003
    eps = np.linspace(0, 0.20, 20000)
    base = np.where(eps <= ey, E * eps, 300 + 150 * (1 - np.exp(-(eps - ey) / 0.03)))
    base = base - 12500 * np.maximum(0, eps - 0.12) ** 2
    sig = np.where(eps < toe, E * eps**2 / (2 * toe), base)
    epsm = np.where(eps < toe, eps, eps + toe / 2)
    load = sig * AREA_MM2 + rng.normal(0, 20, eps.size) + 15      # +15 N offset
    disp = epsm * GAUGE_LENGTH_MM + 0.02                          # +0.02 mm offset
    load = np.r_[load, np.linspace(load[-1] - 300, 10, 8)]        # fracture drop
    disp = np.r_[disp, np.linspace(disp[-1], disp[-1] + 0.05, 8)]
    pd.DataFrame({LOAD_COL: load, DISP_COL: disp}).assign(
        **{LOAD_COL: lambda d: d[LOAD_COL].where(d.index != 100)}
    ).to_csv("sample_tensile.csv", index=False)
    csv_path = "sample_tensile.csv"
else:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
print("Using file:", csv_path)

## Run the analysis

In [ ]:
df = process_tensile_file(csv_path, GAUGE_LENGTH_MM, AREA_MM2, n_baseline=BASELINE_POINTS)
props = analyze_tensile_properties(df)

results = pd.Series({
    "Young's modulus (GPa)": round(props.fit.modulus_gpa, 1),
    "Linear fit R^2": round(props.fit.r_squared, 5),
    "0.2% offset yield strength (MPa)": round(props.yield_strength_mpa, 1),
    "UTS (MPa)": round(props.uts_mpa, 1),
    "Uniform elongation at UTS (%)": round(props.uts_strain * 100, 2),
    "Fracture stress (MPa)": round(props.fracture_stress_mpa, 1),
    "Total elongation at fracture (%)": round(props.elongation_total_pct, 2),
    "Plastic elongation after fracture (%)": round(props.elongation_plastic_pct, 2),
}, name="Value")
results.to_frame()

## Plot, save and download

In [ ]:
fig, ax = plot_stress_strain(df, props, save_path="stress_strain.png", show=True)
fig.savefig("stress_strain.pdf", bbox_inches="tight")   # vector version for papers

In [ ]:
#@title Download results (plot + cleaned data)
df.to_csv("cleaned_stress_strain.csv", index=False)
try:
    from google.colab import files
    for f in ["stress_strain.png", "stress_strain.pdf", "cleaned_stress_strain.csv"]:
        files.download(f)
except ImportError:
    print("Not running in Colab; files saved to the working directory.")